## Установка Gradio

In [ ]:
!pip install gradio -q

In [ ]:
print(gr.__version__)

## Создание окна

In [1]:
import gradio as gr
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import os
from PIL import Image


In [2]:
# Пытаемся подключить диск только если мы в Colab
try:
    from google.colab import drive
    if os.path.exists('/content'):
        if not os.path.isdir('/content/drive'):
            drive.mount('/content/drive')
except ImportError:
    # Если мы локально, библиотеки google.colab нет, и это нормально
    print("Запуск локально: модуль google.colab не найден, пропускаем монтирование.")

Mounted at /content/drive


In [3]:
# Определяем корень проекта (BASE_DIR)
if os.path.exists('/content/drive/MyDrive/'):
    # Путь к папке на Диске (проверьте название 'my_project' или 'CLASSIFICATION')
    BASE_DIR = '/content/drive/MyDrive/CLASSIFICATION' 
else:
    # Локально или на GitHub корень — это текущая папка
    BASE_DIR = '.'

# Ко всем путям приклеиваем BASE_DIR
PATHS = {
    'train': os.path.join(BASE_DIR, 'dataset/train'),
    'validation':   os.path.join(BASE_DIR, 'dataset/validation'),
    'best_model': os.path.join(BASE_DIR, 'best_model.pth')
}

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = ['adidas', 'converse', 'nike']
device

device(type='cpu')

### Загрузка модели

In [5]:
model = models.resnet18()
model.fc = nn.Linear(model.fc.in_features, len(classes))

### Загрузка весов

In [6]:
checkpoint = torch.load(PATHS['best_model'], map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### Трансформация

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

### Интерфейс

In [8]:
def predict_gradio(input_img):
    # Превращаем массив в PIL Image (Gradio подает картинку как numpy array)
    img = Image.fromarray(input_img).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
    
    # Возвращаем словарь {Класс: Вероятность} для красивого отображения
    return {classes[i]: float(probs[i]) for i in range(len(classes))}

### Запуск

In [9]:
demo = gr.Interface(
    fn=predict_gradio, 
    inputs=gr.Image(), 
    outputs=gr.Label(num_top_classes=3),
    title="Классификатор кроссовок",
    description="Загрузи фото кроссовок (Adidas, Nike или Converse), и нейросеть определит бренд."
)

demo.launch(share=True, debug=True, inline=True) # share=True создаст публичную ссылку на 72 часа

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f89159eabb250c56cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f89159eabb250c56cd.gradio.live
